# Lindblad Dynamics via Stochastic Magnus Expansion

This notebook demonstrates the variational quantum simulation of open quantum systems described in

> J.-C. Huang, H.-E. Li, Y.-C. Wang, G.-Z. Zhang, J. Li, H.-S. Hu, *Towards Robust Variational Quantum Simulation of Lindblad Dynamics via Stochastic Magnus Expansion*, **PRX Quantum** 6, 040312 (2025).

The implementation is built entirely on **pyqpanda3** (variational circuits construction and state-vector simulation) and uses only `numpy`/`scipy` for the stochastic integration, Lindblad Liouvillian reference and plotting. No other quantum computing framework is required.

Key building blocks:

* **`magnus.effective_hamiltonian`** — high-order stochastic Magnus integrators (Scheme I-IV) and the Euler-Maruyama scheme for one step of the quantum state diffusion (QSD) unravelling.
* **`ansatz.HardwareEfficientAnsatz`** — parameterised RX/RZ rotations with parameterised RZZ entanglers that reduce to the identity at `theta=0`.
* **`variational.mclachlan_system`** — McLachlan variational principle for non-Hermitian generators, solved as a real least-squares problem.
* **`lindblad.LindbladMagnusSolver`** — high-level solver with trajectory ensemble averaging.
* **`models`** — ready-to-use TFIM-with-damping, FMO and radical-pair models plus a Liouvillian-based exact solver `mesolve`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from pyqpanda_alg.LindbladMagnus import (
    HardwareEfficientAnsatz, LindbladMagnusSolver, fmo_model, tfim_model,
    mesolve, liouvillian, effective_hamiltonian, sample_wiener_integrals,
)

## 1. Exact Lindblad reference: amplitude damping

We first verify the exact solver `mesolve` on the simplest possible open system — a single amplitude-damping channel — for which the excited-state population decays as $\exp(-\gamma t)$.

In [ ]:
gamma = 0.3
H0 = np.zeros((2, 2), dtype=complex)
c_op = np.array([[0, np.sqrt(gamma)], [0, 0]], dtype=complex)
psi0 = np.array([0, 1.0], dtype=complex)  # excited state |1>
tlist = np.linspace(0, 5, 51)
e_ops = [np.array([[0, 0], [0, 1]], dtype=complex)]  # |1><1|

expect = mesolve(H0, psi0, tlist, [c_op], e_ops)
analytic = np.exp(-gamma * tlist)

plt.figure(figsize=(7, 4))
plt.plot(tlist, expect[0], label='mesolve')
plt.plot(tlist, analytic, '--', label=r'$e^{-\gamma t}$')
plt.xlabel('Time'); plt.ylabel('Excited-state population')
plt.legend(); plt.title('Amplitude damping: exact Liouvillian vs analytic')
plt.show()

## 2. Transverse-field Ising model with damping

The TFIM Hamiltonian is $H = Z_0 Z_1 - \tfrac{1}{2}(X_0 + X_1)$ with two independent amplitude-damping channels ($\gamma = 0.1$). Starting from $|11\rangle$, the system undergoes coherent oscillations while slowly relaxing to $|00\rangle$.

In [ ]:
H, c_ops, e_ops, psi0, labels = tfim_model()
print('Hamiltonian dimension:', H.shape)
print('Labels:', labels)
print('Initial state: |11>')

# Build the variational ansatz: 2 layers of RX-RZ + RZZ entanglers.
ansatz = HardwareEfficientAnsatz(n_qubits=2, layers=2, init_state=psi0)
print(f'Ansatz: {ansatz.n_parameters} parameters')
print('Identity at theta=0:', np.allclose(
    ansatz.get_statevector(np.zeros(ansatz.n_parameters)), psi0))

In [ ]:
times = np.linspace(0, 2.5, 51)
exact = mesolve(H, psi0, times, c_ops, e_ops)

solver = LindbladMagnusSolver(H, c_ops, ansatz,
                              qsd_type='nonlinear',
                              magnus_order=1,
                              integrator='rk4')
result = solver.solve(psi0, times, e_ops, traj_num=30, seed=42)
mean = result.expect
print(f'Max error vs exact: {np.abs(mean - exact).max():.4f}')

In [ ]:
palette = plt.cm.tab10.colors
fig, ax = plt.subplots(figsize=(8, 5))
for i, lab in enumerate(labels):
    ax.plot(times, exact[i], '-', color=palette[i], label=f'Exact {lab}')
    ax.plot(times, mean[i], '--', color=palette[i], label=f'Sim {lab}')
ax.set_xlabel('Time'); ax.set_ylabel('Population')
ax.set_title('TFIM with amplitude damping: Lindblad-Magnus variational simulation')
ax.legend()
plt.show()

## 3. Fenna-Matthews-Olson complex

The FMO complex models excitation energy transfer in a photosynthetic pigment-protein complex. We use the 5-site sub-network of the paper: an excitation starts on site 1 and is transferred through the network to a sink, while being dephased by the protein bath and slowly lost to the ground state.

**Note on variational accuracy.** The FMO Hamiltonian lives in a 5-dimensional subspace of the 3-qubit (8-dim) Hilbert space. A generic hardware-efficient ansatz can visit the unused basis states, which limits the variational accuracy for long-time dynamics. For quantitative FMO studies one should use a problem-specific ansatz (Hamiltonian variational ansatz). The example below uses short time scales where the variational simulation still tracks the exact solution reasonably well.

In [ ]:
H, c_ops, e_ops, psi0, labels = fmo_model()
print(f'FMO H shape: {H.shape}, c_ops: {len(c_ops)}, e_ops: {len(e_ops)}')
print('Observables:', labels)

In [ ]:
ansatz = HardwareEfficientAnsatz(n_qubits=3, layers=2, init_state=psi0)
print(f'Ansatz: {ansatz.n_parameters} parameters')

# Short time scale + fine dt for the qualitative behaviour.
times = np.linspace(0, 30, 31)  # dt = 1.0
exact = mesolve(H, psi0, times, c_ops, e_ops)

solver = LindbladMagnusSolver(H, c_ops, ansatz,
                              qsd_type='nonlinear',
                              magnus_order=1,
                              integrator='rk4')
result = solver.solve(psi0, times, e_ops, traj_num=15, seed=42)
mean = result.expect
print(f'Max error vs exact: {np.abs(mean - exact).max():.4f}')

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for i, lab in enumerate(labels):
    ax.plot(times, exact[i], '-', color=palette[i], label=f'Exact {lab}')
    ax.plot(times, mean[i], '--', color=palette[i], label=f'Sim {lab}')
ax.set_xlabel('Time'); ax.set_ylabel('Population')
ax.set_title('FMO excitation transfer: Lindblad-Magnus variational simulation')
ax.legend()
plt.show()

## 4. Inspecting the stochastic Magnus expansion

The effective Hamiltonian $H_{\mathrm{eff}}$ for a single Magnus step is non-Hermitian: its Hermitian part drives coherent oscillations while the anti-Hermitian part accounts for the dissipative norm change.

In [ ]:
H, c_ops, e_ops, psi0, labels = tfim_model()
dt = 0.1
integ = sample_wiener_integrals(len(c_ops), dt, rng=np.random.RandomState(42))
H_eff = effective_hamiltonian(H, c_ops, dt, magnus_order=1,
                              qsd_type='nonlinear', psi=psi0,
                              integrals=integ)
print('Hermitian part (real-time driver):')
print(np.round(0.5 * (H_eff + H_eff.conj().T), 4))
print('\nAnti-Hermitian part (dissipation):')
print(np.round(-0.5j * (H_eff - H_eff.conj().T), 4))